# LeetCode 2086 — Minimum Number of Food Buckets to Feed the Hamsters bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-food-buckets-to-feed-the-hamsters/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimumBuckets(self, hamsters: str) -> int:
        ...
```

(Bản cũ của đề gọi tham số là `street`; LeetCode truyền theo vị trí nên tên tham số không ảnh hưởng.)

Lộ trình:
1. **Brute force** thử mọi tập ô trống đặt xô — `O(2^k · n)`.
2. **DP trạng thái nhỏ** quét trái → phải: nhớ "ô trước có xô không" và "có chuột đang chờ xô ở ô này không".
3. **Tham lam**: gặp chuột chưa được ăn ⇒ ưu tiên đặt xô **bên phải** (để có thể dùng chung cho chuột kế tiếp), không được thì bên trái.
4. **Công thức đếm mẫu**: `số 'H' − số mẫu "H.H" không chồng nhau`, kèm điều kiện bất khả thi.

## 1. Mục tiêu học

1. Mô hình hóa ràng buộc "mỗi chuột cần ít nhất một xô kề bên" và nhận ra trường hợp **bất khả thi** (chuột bị kẹp giữa hai chuột hoặc giữa chuột và biên).
2. Viết **DP trạng thái hữu hạn** khi quyết định tại mỗi ô chỉ phụ thuộc ô ngay trước — mẫu dùng được cho mọi bài "phủ" trên dãy.
3. Chứng minh lựa chọn tham lam "đặt bên phải trước" bằng **lập luận trao đổi** (exchange argument).
4. Rút gọn thành công thức đếm `'H'` và `"H.H"`, hiểu vì sao `str.count` (không chồng nhau) cho đúng giá trị.
5. Kiểm chứng mọi cách bằng brute force trên **toàn bộ** xâu độ dài nhỏ.

## 2. Đề bài & ví dụ

Cho xâu `hamsters`, `hamsters[i]` là `'H'` (có chuột hamster) hoặc `'.'` (ô trống). Ta đặt **xô thức ăn** vào một số ô **trống**. Chuột ở ô `i` được ăn nếu có xô ở ô `i - 1` **hoặc** ô `i + 1`. Trả về số xô **ít nhất** để mọi chuột được ăn, hoặc `-1` nếu không thể.

**Ví dụ 1:** `"H..H"` → `2` (xô ở ô 1 và 2; một xô không đủ cho cả hai chuột).

**Ví dụ 2:** `".H.H."` → `1` (một xô ở ô 2 cho cả hai chuột).

**Ví dụ 3:** `".HHH."` → `-1` (chuột ở giữa không có ô trống kề).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, mỗi ký tự là `'H'` hoặc `'.'`.

In [1]:
import random
import re
import time
from itertools import combinations, product
from typing import List

examples = [("H..H", 2), (".H.H.", 1), (".HHH.", -1)]
for s, want in examples:
    print(f"{s!r} -> kỳ vọng {want}")

'H..H' -> kỳ vọng 2
'.H.H.' -> kỳ vọng 1
'.HHH.' -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1 Kiểm tra một cách đặt xô có hợp lệ không

Viết hàm kiểm tra trước — nó dùng cho brute force và cho test.

In [2]:
def all_fed(s, buckets):
    """buckets: tập chỉ số đặt xô (phải là ô trống)."""
    if any(s[b] != "." for b in buckets):
        return False
    return all((i - 1 in buckets) or (i + 1 in buckets) for i, c in enumerate(s) if c == "H")

print(all_fed("H..H", {1, 2}), all_fed("H..H", {1}), all_fed(".H.H.", {2}), all_fed(".H.H.", {0, 4}))

True False True True


### 3.2 `combinations` để thử mọi tập cỡ `k`

Duyệt `k = 0, 1, 2, …` và dừng ở `k` đầu tiên có tập hợp lệ ⇒ tìm được **số nhỏ nhất**.

In [3]:
empty = [i for i, c in enumerate(".H.H.") if c == "."]
print("ô trống:", empty)
print("các tập cỡ 1:", list(combinations(empty, 1)))

ô trống: [0, 2, 4]
các tập cỡ 1: [(0,), (2,), (4,)]


### 3.3 `str.count` đếm mẫu **không chồng nhau**

`"H.H.H".count("H.H")` trả về `1`, không phải `2`: sau khi khớp ở vị trí 0, việc tìm tiếp bắt đầu **sau** đoạn đã khớp. Muốn đếm chồng nhau phải dùng regex lookahead.

In [4]:
print("không chồng nhau:", "H.H.H".count("H.H"), "| chồng nhau:", len(re.findall(r"(?=H\.H)", "H.H.H")))

không chồng nhau: 1 | chồng nhau: 2


### 3.4 DP với vài trạng thái — dùng `dict` hoặc biến riêng

Khi trạng thái chỉ có vài giá trị, lưu `dp` dạng `dict {trạng_thái: chi_phí_nhỏ_nhất}` và cập nhật sang `ndp` ở mỗi bước.

## 4. Ý tưởng nền tảng / trực giác

**Khi nào bất khả thi?** Chuột ở `i` cần `s[i-1] == '.'` hoặc `s[i+1] == '.'`. Nếu cả hai phía là chuột hoặc biên ⇒ `-1`. Ví dụ: `"HHH"` (chuột giữa), `"HH."` (chuột ở ô 0: trái là biên, phải là chuột), `"H"` (một chuột đơn độc).

**Khi nào một xô phục vụ được hai chuột?** Chỉ khi xô nằm giữa mẫu `"H.H"`. Mọi xô khác chỉ phục vụ tối đa một chuột.

Tính tay:

| xâu | phân tích | đáp án |
|---|---|---|
| `"H..H"` | không có `"H.H"`; mỗi chuột một xô | 2 |
| `".H.H."` | một mẫu `"H.H"` ở ô 1..3 ⇒ chung xô ô 2 | 1 |
| `"H.H.H"` | hai mẫu chồng nhau, chỉ dùng được một (ô 1); chuột ô 4 cần xô ô 3 | 2 |
| `"H.HH.H"` | ô 0..2 là `"H.H"` (xô ô 1); chuột ô 3, 5 chung xô ô 4 | 2 |
| `".HHH."` | chuột ô 2 bị kẹp | -1 |

In [5]:
for s in ["H..H", ".H.H.", "H.H.H", "H.HH.H", ".HHH."]:
    stuck = [i for i, c in enumerate(s) if c == "H"
             and (i == 0 or s[i - 1] == "H") and (i == len(s) - 1 or s[i + 1] == "H")]
    print(f"{s!r}: số H = {s.count('H')}, mẫu H.H (không chồng) = {s.count('H.H')}, chuột bị kẹp = {stuck}")

'H..H': số H = 2, mẫu H.H (không chồng) = 0, chuột bị kẹp = []
'.H.H.': số H = 2, mẫu H.H (không chồng) = 1, chuột bị kẹp = []
'H.H.H': số H = 3, mẫu H.H (không chồng) = 1, chuột bị kẹp = []
'H.HH.H': số H = 4, mẫu H.H (không chồng) = 2, chuột bị kẹp = []
'.HHH.': số H = 3, mẫu H.H (không chồng) = 0, chuột bị kẹp = [2]


## 5. Cách 1 — Brute Force / baseline: thử mọi tập ô trống

**Ý tưởng:** với `k = 0, 1, 2, …`, thử mọi tập `k` ô trống; tập đầu tiên làm mọi chuột được ăn cho đáp án `k`. Nếu thử hết mà không có ⇒ `-1` (chỉ cần thử tập **mọi** ô trống: nếu nó không được thì không tập nào được).

In [6]:
def buckets_brute(s):
    empty = [i for i, c in enumerate(s) if c == "."]
    if not all_fed(s, set(empty)):
        return -1                                  # đặt ở mọi ô trống vẫn không đủ
    for k in range(len(empty) + 1):
        for comb in combinations(empty, k):
            if all_fed(s, set(comb)):
                return k

for s, want in examples:
    print(buckets_brute(s), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
-1 kỳ vọng -1


**Độ phức tạp:** tệ nhất `O(2^k · n)` với `k` = số ô trống — chỉ dùng được với `n ≤ ~20`. Space `O(n)`.

**Nút thắt:** quyết định ở ô `i` chỉ ảnh hưởng chuột ở `i - 1` và `i + 1` — **tính cục bộ** này cho phép quét một lượt thay vì thử mọi tổ hợp.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DP):** quét trái → phải; khi đứng ở ô `i`, quá khứ chỉ còn ảnh hưởng qua hai bit:
- `b` — ô `i - 1` có xô không (để biết chuột ở `i` đã được ăn từ bên trái chưa);
- `need` — chuột ở `i - 1` **chưa** được ăn và **bắt buộc** xô ở ô `i`.

**Insight 2 (tham lam):** gặp chuột ở `i` **chưa được ăn** (ô `i - 1` không có xô):
- *nếu* ô `i + 1` trống *thì* đặt xô ở `i + 1` (xô bên phải có thể phục vụ thêm chuột ở `i + 2`; xô bên trái thì không giúp ai thêm vì mọi chuột bên trái đã được xử lý);
- *ngược lại nếu* ô `i - 1` trống *thì* đặt ở `i - 1`;
- *ngược lại* ⇒ `-1`.

**Insight 3 (công thức):** mỗi xô phục vụ 1 hoặc 2 chuột; phục vụ 2 chỉ khi ở giữa `"H.H"`. Tham lam ghép được đúng số mẫu `"H.H"` **không chồng nhau** tính từ trái ⇒ đáp án `= #H − #("H.H" không chồng nhau)` khi khả thi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP hai bit trạng thái

In [7]:
def buckets_dp(s):
    INF = float("inf")
    dp = {(False, False): 0}                       # (b: ô trước có xô, need: chuột ô trước chờ xô ở ô này)
    for c in s:
        ndp = {}
        def relax(state, cost):
            if cost < ndp.get(state, INF):
                ndp[state] = cost
        for (b, need), cost in dp.items():
            if c == "H":
                if need:
                    continue                       # chuột trước cần xô ở đây nhưng ô này là chuột
                relax((False, not b), cost)        # chưa ăn được từ trái ⇒ cần xô ở ô kế tiếp
            else:
                relax((True, False), cost + 1)     # đặt xô
                if not need:
                    relax((False, False), cost)    # không đặt (chỉ được nếu không ai chờ)
        dp = ndp
        if not dp:
            return -1
    best = min((cost for (b, need), cost in dp.items() if not need), default=INF)
    return -1 if best == INF else best

for s, want in examples:
    print(buckets_dp(s), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
-1 kỳ vọng -1


**Độ phức tạp:** `O(n · 4)` = `O(n)` thời gian, `O(1)` bộ nhớ (tối đa 4 trạng thái). DP không cần "nghĩ ra" quy tắc tham lam — an toàn khi biến thể đổi luật (xô có giá khác nhau, xô phục vụ bán kính 2…).

### 7.2 Cách 3 — Tham lam ưu tiên bên phải (code nộp)

In [8]:
def buckets_greedy(s):
    n = len(s)
    placed = [False] * n
    ans = 0
    for i, c in enumerate(s):
        if c != "H":
            continue
        if i > 0 and placed[i - 1]:
            continue                               # đã được ăn từ bên trái
        if i + 1 < n and s[i + 1] == ".":
            placed[i + 1] = True; ans += 1         # ưu tiên bên phải
        elif i > 0 and s[i - 1] == ".":
            placed[i - 1] = True; ans += 1
        else:
            return -1
    return ans

for s, want in examples:
    print(buckets_greedy(s), "kỳ vọng", want)

2 kỳ vọng 2
1 kỳ vọng 1
-1 kỳ vọng -1


Phiên bản không cần mảng `placed`: khi đặt xô ở `i + 1`, chuột ở `i + 2` (nếu có) chắc chắn được ăn ⇒ nhảy thẳng `i += 3`.

Code nộp LeetCode của Cách 3 (bản nhảy chỉ số): xem mục 9.

**Độ phức tạp:** `O(n)` thời gian, `O(n)` (hoặc `O(1)` với bản nhảy chỉ số).

### 7.3 Cách 4 — Công thức đếm mẫu

In [9]:
def buckets_formula(s):
    n = len(s)
    for i, c in enumerate(s):                      # điều kiện bất khả thi
        if c == "H" and (i == 0 or s[i - 1] == "H") and (i == n - 1 or s[i + 1] == "H"):
            return -1
    return s.count("H") - s.count("H.H")

def buckets_formula_short(s):
    if s == "H" or "HHH" in s or s.startswith("HH") or s.endswith("HH"):
        return -1
    return s.count("H") - s.count("H.H")

for s, want in examples:
    print(buckets_formula(s), buckets_formula_short(s), "kỳ vọng", want)

2 2 kỳ vọng 2
1 1 kỳ vọng 1
-1 -1 kỳ vọng -1


**Độ phức tạp:** `O(n)` (vài lần quét bằng hàm C), `O(1)` bộ nhớ.

Bản rút gọn `buckets_formula_short` dựa trên nhận xét: chuột bị kẹp ⇔ nó là ô giữa của `"HHH"`, hoặc là ô đầu của xâu bắt đầu bằng `"HH"`, hoặc ô cuối của xâu kết thúc bằng `"HH"`, hoặc xâu chỉ là `"H"`. Mục 12 kiểm chứng cả hai bản trên mọi xâu độ dài ≤ 14.

## 8. TRACE từng bước

Trace tham lam trên `"H.H.H"`, `"H.HH.H"` và `"..H.HH.H.."`: in chuột đang xét, đã được ăn chưa, đặt xô ở đâu.

In [10]:
def trace_greedy(s):
    n = len(s); placed = [False] * n; ans = 0
    print(f"s = {s!r}")
    for i, c in enumerate(s):
        if c != "H":
            continue
        if i > 0 and placed[i - 1]:
            print(f"  chuột {i}: đã có xô ở {i - 1} -> bỏ qua"); continue
        if i + 1 < n and s[i + 1] == ".":
            placed[i + 1] = True; ans += 1; print(f"  chuột {i}: đặt xô PHẢI ở {i + 1} (tổng {ans})")
        elif i > 0 and s[i - 1] == ".":
            placed[i - 1] = True; ans += 1; print(f"  chuột {i}: bên phải không trống -> đặt xô TRÁI ở {i - 1} (tổng {ans})")
        else:
            print(f"  chuột {i}: bị kẹp -> -1"); return -1
    print("  bố trí:", "".join("B" if placed[k] else s[k] for k in range(n)), "->", ans)
    return ans

for s in ["H.H.H", "H.HH.H", "..H.HH.H.."]:
    trace_greedy(s)

s = 'H.H.H'
  chuột 0: đặt xô PHẢI ở 1 (tổng 1)
  chuột 2: đã có xô ở 1 -> bỏ qua
  chuột 4: bên phải không trống -> đặt xô TRÁI ở 3 (tổng 2)
  bố trí: HBHBH -> 2
s = 'H.HH.H'
  chuột 0: đặt xô PHẢI ở 1 (tổng 1)
  chuột 2: đã có xô ở 1 -> bỏ qua
  chuột 3: đặt xô PHẢI ở 4 (tổng 2)
  chuột 5: đã có xô ở 4 -> bỏ qua
  bố trí: HBHHBH -> 2
s = '..H.HH.H..'
  chuột 2: đặt xô PHẢI ở 3 (tổng 1)
  chuột 4: đã có xô ở 3 -> bỏ qua
  chuột 5: đặt xô PHẢI ở 6 (tổng 2)
  chuột 7: đã có xô ở 6 -> bỏ qua
  bố trí: ..HBHHBH.. -> 2


In [11]:
def trace_dp(s):
    dp = {(False, False): 0}
    print(f"s = {s!r}; trạng thái (b, need): chi phí")
    for i, c in enumerate(s):
        ndp = {}
        for (b, need), cost in dp.items():
            if c == "H":
                if not need:
                    st = (False, not b); ndp[st] = min(ndp.get(st, 10**9), cost)
            else:
                ndp[(True, False)] = min(ndp.get((True, False), 10**9), cost + 1)
                if not need:
                    ndp[(False, False)] = min(ndp.get((False, False), 10**9), cost)
        dp = ndp
        print(f"  sau ô {i} ({c}):", {f"b={int(k[0])},need={int(k[1])}": v for k, v in sorted(dp.items())})
    print("  đáp án:", min((v for k, v in dp.items() if not k[1]), default=-1))

trace_dp(".H.H.")

s = '.H.H.'; trạng thái (b, need): chi phí
  sau ô 0 (.): {'b=0,need=0': 0, 'b=1,need=0': 1}
  sau ô 1 (H): {'b=0,need=0': 1, 'b=0,need=1': 0}
  sau ô 2 (.): {'b=0,need=0': 1, 'b=1,need=0': 1}
  sau ô 3 (H): {'b=0,need=0': 1, 'b=0,need=1': 1}
  sau ô 4 (.): {'b=0,need=0': 1, 'b=1,need=0': 2}
  đáp án: 1


Đọc trace DP `".H.H."`: sau ô 1 (`H`), trạng thái `need = 1` chi phí 0 nghĩa là "chưa xô nào, chuột ô 1 đang chờ xô ở ô 2". Ở ô 2 bắt buộc đặt ⇒ `(b=1, need=0)` chi phí 1. Ô 3 (`H`) từ `b = 1` ⇒ đã ăn ⇒ `need = 0`. Kết thúc với chi phí nhỏ nhất `1`. ✓

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minimumBuckets(self, hamsters: str) -> int:
        n = len(hamsters)
        ans = 0
        i = 0
        while i < n:
            if hamsters[i] == "H":
                if i + 1 < n and hamsters[i + 1] == ".":
                    ans += 1
                    i += 3                # xô ở i+1 cũng nuôi chuột ở i+2 (nếu có) ⇒ bỏ qua i+1, i+2
                    continue
                if i > 0 and hamsters[i - 1] == ".":
                    ans += 1              # i-1 trống và chắc chắn chưa có xô (nếu có, ta đã nhảy qua i)
                else:
                    return -1
            i += 1
        return ans

sol = Solution()
for s, want in examples:
    got = sol.minimumBuckets(s)
    print(got, "OK" if got == want else "SAI")

2 OK
1 OK
-1 OK


**Vì sao `i += 3` an toàn?** Đặt xô ở `i + 1`. Ô `i + 1` là xô (không có chuột). Ô `i + 2` nếu là chuột thì đã được ăn nhờ xô ở `i + 1`. Vậy vòng lặp có thể tiếp tục từ `i + 3`. Và khi một chuột ở `j` được xét đến (không bị nhảy qua), ô `j - 1` chắc chắn **chưa** có xô — vì mọi xô ta đặt bên phải đều kèm theo cú nhảy vượt qua chuột kế bên.

## 10. Vì sao thuật toán đúng

**Điều kiện khả thi.** Nếu một chuột không có ô trống kề thì không cách nào nuôi nó ⇒ `-1`. Ngược lại, đặt xô ở **mọi** ô trống là một lời giải ⇒ khả thi. Tham lam trả `-1` đúng khi gặp chuột như vậy.

**Tối ưu của tham lam (lập luận trao đổi).** Xét chuột `i` đầu tiên chưa được ăn khi quét. Mọi lời giải tối ưu `OPT` phải có xô ở `i - 1` hoặc `i + 1`.
- Nếu `s[i+1] = '.'` và `OPT` dùng `i - 1` (không dùng `i + 1`): xô ở `i - 1` chỉ phục vụ chuột `i` và có thể chuột `i - 2` — nhưng chuột `i - 2` (nếu có) đã được xử lý trước đó và tham lam đã nuôi nó bằng xô khác (ta có thể giả sử `OPT` khớp tham lam trên phần đã xét, theo quy nạp). Dời xô từ `i - 1` sang `i + 1`: chuột `i` vẫn được ăn, chuột `i + 2` có thể được thêm, không ai bị mất ⇒ vẫn hợp lệ, cùng số xô. Vậy tồn tại tối ưu chứa `i + 1`.
- Nếu `s[i+1]` không trống, bắt buộc dùng `i - 1`.

Quy nạp theo thứ tự chuột ⇒ tham lam đạt số xô của một lời giải tối ưu. ✓

**Công thức.** Theo tham lam, mỗi xô đặt bên phải ở `i + 1` nuôi thêm chuột `i + 2` khi và chỉ khi `s[i..i+2] = "H.H"`, và sau đó quét tiếp từ `i + 3` — đúng cơ chế "tìm mẫu không chồng nhau từ trái sang" của `str.count`. Mọi chuột còn lại dùng riêng một xô. ⇒ `#xô = #H − #"H.H" (không chồng)`. ✓

**DP.** Hai bit `(b, need)` mô tả đầy đủ ảnh hưởng của quá khứ lên tương lai (chuột ở `i` chỉ phụ thuộc ô `i - 1` và `i + 1`) ⇒ DP duyệt mọi lời giải hợp lệ một cách nén. ✓

## 11. Phân tích Time / Space

`n = len(hamsters) ≤ 10⁵`, `k` = số ô trống.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force tập con | `O(2^k · n)` | `O(n)` |
| 2. DP hai bit | `O(n)` | `O(1)` |
| 3. Tham lam | `O(n)` | `O(1)` |
| 4. Công thức | `O(n)` | `O(1)` |

In [13]:
rng = random.Random(1)
big = [".H." * 33333 + ".", "".join(rng.choice("H..") for _ in range(10**5)), "H." * 50000]
for name, fn in [("dp", buckets_dp), ("greedy", buckets_greedy), ("formula", buckets_formula),
                 ("formula_short", buckets_formula_short), ("submit", Solution().minimumBuckets)]:
    t0 = time.perf_counter(); r = [fn(x) for x in big]
    print(f"{name:13s}: {r}  {(time.perf_counter() - t0) * 1000:.1f} ms")

dp           : [33333, -1, 25000]  70.4 ms
greedy       : [33333, -1, 25000]  9.4 ms
formula      : [33333, -1, 25000]  6.5 ms
formula_short: [33333, -1, 25000]  0.3 ms
submit       : [33333, -1, 25000]  4.5 ms


## 12. Bộ test

Brute force trên **mọi** xâu độ dài `1..14` (`2 + 4 + … + 2¹⁴ ≈ 32 000` xâu) — kiểm chứng toàn diện cả điều kiện `-1` lẫn giá trị tối ưu.

In [14]:
fns = [buckets_dp, buckets_greedy, buckets_formula, buckets_formula_short, Solution().minimumBuckets]

for s, want in examples:
    assert buckets_brute(s) == want
    for fn in fns:
        assert fn(s) == want

edge = [("H", -1), (".", 0), ("H.", 1), (".H", 1), ("HH", -1), ("HH.", -1), (".HH", -1), ("H.H", 1),
        ("H.H.H", 2), ("H.HH.H", 2), ("....", 0), (".H.H.H.H.", 2), (".HH.HH.", 3)]
for s, want in edge:
    assert buckets_brute(s) == want, s
    for fn in fns:
        assert fn(s) == want, (fn, s)

checked = 0
for n in range(1, 15):
    for t in product("H.", repeat=n):
        s = "".join(t)
        want = buckets_brute(s)
        for fn in fns:
            assert fn(s) == want, (fn, s)
        checked += 1
print(f"ví dụ + edge + toàn bộ {checked} xâu độ dài 1..14 khớp brute force")

ví dụ + edge + toàn bộ 32766 xâu độ dài 1..14 khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam ưu tiên bên **trái**

In [15]:
def wrong_left_first(s):
    n = len(s); placed = [False] * n; ans = 0
    for i, c in enumerate(s):
        if c != "H" or (i > 0 and placed[i - 1]):
            continue
        if i > 0 and s[i - 1] == ".":                         # SAI: ưu tiên trái
            placed[i - 1] = True; ans += 1
        elif i + 1 < n and s[i + 1] == ".":
            placed[i + 1] = True; ans += 1
        else:
            return -1
    return ans

for s, want in examples + [("H.H.H", 2)]:
    print(f"{s!r}: sai = {wrong_left_first(s)} | đúng = {want}")

'H..H': sai = 2 | đúng = 2
'.H.H.': sai = 2 | đúng = 1
'.HHH.': sai = -1 | đúng = -1
'H.H.H': sai = 2 | đúng = 2


Xô bên trái không giúp được chuột nào khác (bên trái đã xử lý xong), còn xô bên phải có thể dùng chung. `".H.H."`: ưu tiên trái đặt ở ô 0 và ô 2 ⇒ `2` thay vì `1`. **Sửa:** ưu tiên phải.

### Lỗi 2 — Không kiểm tra chuột đã được ăn từ xô trước

In [16]:
def wrong_no_fed_check(s):
    n = len(s); ans = 0
    for i, c in enumerate(s):
        if c != "H":
            continue
        if i + 1 < n and s[i + 1] == ".":                     # SAI: chuột ô i có thể đã ăn nhờ xô ở i-1
            ans += 1
        elif i > 0 and s[i - 1] == ".":
            ans += 1
        else:
            return -1
    return ans

for s, want in examples + [("H.H", 1)]:
    print(f"{s!r}: sai = {wrong_no_fed_check(s)} | đúng = {want}")

'H..H': sai = 2 | đúng = 2
'.H.H.': sai = 2 | đúng = 1
'.HHH.': sai = -1 | đúng = -1
'H.H': sai = 2 | đúng = 1


Mỗi chuột tự đặt một xô ⇒ đáp án luôn bằng số chuột. **Sửa:** nhớ ô nào đã có xô (mảng `placed`) hoặc nhảy chỉ số `i += 3`.

### Lỗi 3 — Điều kiện `-1` chỉ kiểm tra `"HHH"` (quên biên)

In [17]:
def wrong_impossible_check(s):
    if "HHH" in s:                                            # SAI: thiếu "HH" ở đầu/cuối và "H" đơn
        return -1
    return s.count("H") - s.count("H.H")

for s, want in [("HH.", -1), (".HH", -1), ("H", -1), (".HHH.", -1), ("H.H", 1)]:
    print(f"{s!r}: sai = {wrong_impossible_check(s)} | đúng = {want}")

'HH.': sai = 2 | đúng = -1
'.HH': sai = 2 | đúng = -1
'H': sai = 1 | đúng = -1
'.HHH.': sai = -1 | đúng = -1
'H.H': sai = 1 | đúng = 1


Biên xâu cũng "chặn" như một con chuột. **Sửa:** kiểm tra từng chuột có ô trống kề không (bản an toàn), hoặc đủ bốn điều kiện của bản rút gọn.

### Lỗi 4 — Đếm mẫu `"H.H"` **chồng nhau**

In [18]:
def wrong_overlap(s):
    if buckets_formula(s) == -1:
        return -1
    return s.count("H") - len(re.findall(r"(?=H\.H)", s))     # SAI: một xô không phục vụ 3 chuột

for s, want in [("H.H.H", 2), ("H.H.H.H", 2), (".H.H.", 1)]:
    print(f"{s!r}: sai = {wrong_overlap(s)} | đúng = {want}")

'H.H.H': sai = 1 | đúng = 2
'H.H.H.H': sai = 1 | đúng = 2
'.H.H.': sai = 1 | đúng = 1


`"H.H.H"` có hai mẫu chồng nhau nhưng chuột ở giữa chỉ cần **một** xô; ghép cả hai sẽ coi như 2 xô nuôi 3 chuột với một xô "dùng chung" hai lần. **Sửa:** đếm không chồng nhau (`str.count`).

### Lỗi 5 — Đặt xô lên ô có chuột

Trong tham lam, nếu chỉ kiểm tra `i + 1 < n` mà quên `s[i + 1] == '.'`, ta sẽ "đặt xô" lên chuột ⇒ `"HH."` cho `1` thay vì `-1`.

In [19]:
def wrong_on_hamster(s):
    n = len(s); placed = [False] * n; ans = 0
    for i, c in enumerate(s):
        if c != "H" or (i > 0 and placed[i - 1]):
            continue
        if i + 1 < n:                                          # SAI: không kiểm tra ô trống
            placed[i + 1] = True; ans += 1
        elif i > 0 and s[i - 1] == ".":
            placed[i - 1] = True; ans += 1
        else:
            return -1
    return ans

for s, want in [("HH.", -1), (".HHH.", -1), (".HH.HH.", 3)]:
    print(f"{s!r}: sai = {wrong_on_hamster(s)} | đúng = {want}")

'HH.': sai = 2 | đúng = -1
'.HHH.': sai = 2 | đúng = -1
'.HH.HH.': sai = 3 | đúng = 3


**Sửa:** chỉ đặt vào ô `'.'`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phủ" các điểm trên dãy bằng vật đặt ở ô kề, tối thiểu số vật ⇒ **tham lam đặt càng xa về phía chưa xét càng tốt** (tương tự đặt trạm phủ khoảng).
- Lập luận trao đổi: "dời vật sang phải không làm mất gì, có thể được thêm".
- Không chắc tham lam đúng ⇒ viết **DP trạng thái nhỏ** (2–4 trạng thái) và so sánh trên mọi xâu ngắn.
- Biên xâu đóng vai trò như vật cản — luôn thử `"H"`, `"HH."`, `".HH"`.
- Họ bài: 968 *Binary Tree Cameras* (phủ trên cây, cùng tư tưởng), 2027 *Minimum Moves to Convert String*, 1326 *Minimum Number of Taps to Open to Water a Garden*, 605 *Can Place Flowers*.

## 15. Biến thể của bài

1. **Đường phố vòng tròn** (ô `0` kề ô `n - 1`): xử lý bằng cách thử hai trường hợp cho ô đầu (Bài 5).
2. **Xô phục vụ bán kính `r`** (chuột ở `i` ăn được nếu có xô trong `[i - r, i + r]`): tham lam "đặt xô ở ô trống xa nhất về bên phải trong tầm" vẫn đúng.
3. **Mỗi ô có chi phí đặt xô khác nhau**: tham lam hỏng, DP hai bit vẫn chạy (thay `+1` bằng `+cost[i]`).
4. **Đếm số cách đặt tối ưu**: DP lưu cặp (chi phí nhỏ nhất, số cách).

In [20]:
def buckets_weighted_dp(s, cost):
    """Biến thể 3: DP hai bit với chi phí từng ô."""
    INF = float("inf"); dp = {(False, False): 0}
    for i, c in enumerate(s):
        ndp = {}
        for (b, need), v in dp.items():
            if c == "H":
                if not need:
                    st = (False, not b); ndp[st] = min(ndp.get(st, INF), v)
            else:
                ndp[(True, False)] = min(ndp.get((True, False), INF), v + cost[i])
                if not need:
                    ndp[(False, False)] = min(ndp.get((False, False), INF), v)
        dp = ndp
    best = min((v for (b, need), v in dp.items() if not need), default=INF)
    return -1 if best == INF else best

def buckets_weighted_brute(s, cost):
    empty = [i for i, c in enumerate(s) if c == "."]; best = None
    for k in range(len(empty) + 1):
        for comb in combinations(empty, k):
            if all_fed(s, set(comb)):
                v = sum(cost[i] for i in comb); best = v if best is None else min(best, v)
    return -1 if best is None else best

def buckets_radius(s, r):
    """Biến thể 2: tham lam đặt xô ở ô trống xa nhất bên phải trong tầm của chuột chưa ăn."""
    n = len(s); last = -10**9; ans = 0
    for i, c in enumerate(s):
        if c != "H" or i - last <= r:
            continue
        for j in range(min(n - 1, i + r), i - r - 1, -1):
            if j >= 0 and s[j] == "." and abs(j - i) <= r:
                last = j; ans += 1; break
        else:
            return -1
    return ans

def buckets_radius_brute(s, r):
    empty = [i for i, c in enumerate(s) if c == "."]
    for k in range(len(empty) + 1):
        for comb in combinations(empty, k):
            if all(any(abs(b - i) <= r for b in comb) for i, c in enumerate(s) if c == "H"):
                return k
    return -1

rng = random.Random(15)
for _ in range(400):
    s = "".join(rng.choice("H..") for _ in range(rng.randint(1, 11)))
    cost = [rng.randint(1, 5) for _ in s]
    assert buckets_weighted_dp(s, cost) == buckets_weighted_brute(s, cost)
    rr = rng.randint(1, 3)
    assert buckets_radius(s, rr) == buckets_radius_brute(s, rr), (s, rr)
print("biến thể chi phí (DP) và bán kính r (tham lam) khớp brute force")
print("ví dụ chi phí: '.H.H.' với cost [1, 0, 5, 0, 1] ->", buckets_weighted_dp(".H.H.", [1, 0, 5, 0, 1]), "(2 xô rẻ tốt hơn 1 xô đắt)")

biến thể chi phí (DP) và bán kính r (tham lam) khớp brute force
ví dụ chi phí: '.H.H.' với cost [1, 0, 5, 0, 1] -> 2 (2 xô rẻ tốt hơn 1 xô đắt)


Ví dụ cuối cho thấy vì sao tham lam không dùng được khi có chi phí: một xô giữa (`5`) đắt hơn hai xô biên (`1 + 1`).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử tập con | `O(2^k n)` | `O(n)` | kiểm chứng |
| 2. DP hai bit | trạng thái cục bộ | `O(n)` | `O(1)` | biến thể có chi phí / luật lạ |
| 3. Tham lam phải trước | trao đổi | `O(n)` | `O(1)` | **nộp bài** |
| 4. Công thức | `#H − #"H.H"` | `O(n)` | `O(1)` | ngắn nhất, cần chứng minh |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `"H.H.H.H"`? `"..H.."`? `"H..H..H"`? `".H"`?

**Bài 2 (tính tay):** chạy tham lam trên `".HH.H.HH."`, ghi lại xô đặt ở đâu và đáp án.

**Bài 3 (điền TODO):** hoàn thiện `is_feasible(s)` trả về `True` nếu có thể nuôi mọi chuột (không cần số xô).

**Bài 4 (tự viết lại):** viết `bucket_positions(s)` trả về **danh sách vị trí xô** của lời giải tham lam (hoặc `None` nếu bất khả thi), và kiểm tra bằng `all_fed`.

**Bài 5 (mini challenge):** **đường phố vòng tròn** — ô `0` và ô `n - 1` kề nhau (với `n ≥ 3`). Tìm số xô ít nhất hoặc `-1`. Kiểm chứng bằng brute force. (Biến thể tự đặt — đối chiếu lại đề gốc nếu dùng làm đề.)

In [21]:
def is_feasible(s):
    # TODO: mọi 'H' phải có ít nhất một ô '.' kề (trong phạm vi xâu)
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Kiểm tra bất khả thi trước. (2) Đếm `H` và `"H.H"` không chồng nhau. (3) `4 − 2 = 2`; `1`; `3`; `1`.

**Bài 2:** (1) Chuột ô 1: phải là `H` ⇒ đặt trái ô 0. (2) Chuột ô 2: phải trống ⇒ xô ô 3, nuôi luôn chuột ô 4. (3) Chuột ô 6: xô ô 5; chuột ô 7: chưa ăn (ô 6 không phải xô) ⇒ xô ô 8. Tổng `4`.

**Bài 3:** (1) Một vòng `for`. (2) Điều kiện: `(i > 0 and s[i-1] == '.') or (i + 1 < n and s[i+1] == '.')`. (3) `all(...)`.

**Bài 4:** (1) Tham lam với mảng `placed`. (2) Trả `[i for i in range(n) if placed[i]]`. (3) `len` của danh sách phải bằng đáp án.

**Bài 5:** (1) Nếu không có chuột ⇒ `0`. (2) Xoay xâu để ô cuối là `'.'`? Không phải lúc nào cũng có — thay vào đó: thử hai trường hợp "ô `n - 1` có xô" / "không có xô" nếu ô `n - 1` trống, rồi chạy DP tuyến tính với điều kiện biên tương ứng. (3) Đơn giản hơn: chạy DP hai bit trên xâu tuyến tính nhưng **cố định** trạng thái ô `n - 1` ở đầu và kiểm tra lại ở cuối (chuột ô 0 nhìn thấy ô `n - 1`, chuột ô `n - 1` nhìn thấy ô 0).

## 19. Đáp án tham khảo

In [22]:
# Bài 1
for s, want in [("H.H.H.H", 2), ("..H..", 1), ("H..H..H", 3), (".H", 1)]:
    assert Solution().minimumBuckets(s) == buckets_brute(s) == want

# Bài 2
assert trace_greedy(".HH.H.HH.") == 4 == buckets_brute(".HH.H.HH.")

# Bài 3
def is_feasible_ans(s):
    n = len(s)
    return all((i > 0 and s[i - 1] == ".") or (i + 1 < n and s[i + 1] == ".") for i, c in enumerate(s) if c == "H")

# Bài 4
def bucket_positions_ans(s):
    n = len(s); placed = [False] * n
    for i, c in enumerate(s):
        if c != "H" or (i > 0 and placed[i - 1]):
            continue
        if i + 1 < n and s[i + 1] == ".": placed[i + 1] = True
        elif i > 0 and s[i - 1] == ".": placed[i - 1] = True
        else: return None
    return [i for i in range(n) if placed[i]]

for n in range(1, 12):
    for t in product("H.", repeat=n):
        s = "".join(t)
        assert is_feasible_ans(s) == (buckets_brute(s) != -1)
        pos = bucket_positions_ans(s)
        if pos is None:
            assert buckets_brute(s) == -1
        else:
            assert all_fed(s, set(pos)) and len(pos) == buckets_brute(s)
print("Bài 4 ví dụ:", bucket_positions_ans(".HH.H.HH."))
print("Bài 1-4: OK")

s = '.HH.H.HH.'
  chuột 1: bên phải không trống -> đặt xô TRÁI ở 0 (tổng 1)
  chuột 2: đặt xô PHẢI ở 3 (tổng 2)
  chuột 4: đã có xô ở 3 -> bỏ qua
  chuột 6: bên phải không trống -> đặt xô TRÁI ở 5 (tổng 3)
  chuột 7: đặt xô PHẢI ở 8 (tổng 4)
  bố trí: BHHBHBHHB -> 4
Bài 4 ví dụ: [0, 3, 5, 8]
Bài 1-4: OK


In [23]:
# Bài 5 — đường phố vòng tròn
def circular_brute(s):
    n = len(s); empty = [i for i, c in enumerate(s) if c == "."]
    fed = lambda B: all(((i - 1) % n in B) or ((i + 1) % n in B) for i, c in enumerate(s) if c == "H")
    for k in range(len(empty) + 1):
        for comb in combinations(empty, k):
            if fed(set(comb)):
                return k
    return -1

def circular_rotate(s):
    """Lời giải gọn: nếu có ô trống, xoay xâu để ô trống đứng cuối rồi thử 2 trạng thái của nó."""
    n = len(s)
    if "H" not in s:
        return 0
    if "." not in s:
        return -1
    k = s.index(".")
    t = s[k + 1:] + s[:k + 1]                          # t kết thúc bằng '.', thứ tự vòng giữ nguyên
    INF = float("inf"); best = INF
    for last_has in (False, True):
        dp = {(last_has, False): 0}                    # "ô trước t[0]" là t[-1]
        for i in range(n - 1):
            c = t[i]; ndp = {}
            for (b, need), v in dp.items():
                if c == "H":
                    if not need:
                        st = (False, not b); ndp[st] = min(ndp.get(st, INF), v)
                else:
                    ndp[(True, False)] = min(ndp.get((True, False), INF), v + 1)
                    if not need:
                        ndp[(False, False)] = min(ndp.get((False, False), INF), v)
            dp = ndp
        for (b, need), v in dp.items():                # ô cuối t[-1] = '.', trạng thái = last_has
            if need and not last_has:
                continue                               # chuột t[n-2] cần xô ở t[n-1]
            best = min(best, v + (1 if last_has else 0))
    return -1 if best == INF else best

rng = random.Random(55)
for n in range(3, 12):
    for t in product("H.", repeat=n):
        s = "".join(t)
        assert circular_rotate(s) == circular_brute(s), s
print("Bài 5: vòng tròn — DP sau khi xoay khớp brute force trên mọi xâu độ dài 3..11")
print("ví dụ:", [(s, circular_rotate(s)) for s in ["H.H", "HH.", "H..H", ".HHH."]])

Bài 5: vòng tròn — DP sau khi xoay khớp brute force trên mọi xâu độ dài 3..11
ví dụ: [('H.H', 1), ('HH.', 1), ('H..H', 2), ('.HHH.', -1)]


Mẹo ở Bài 5: **xoay** vòng tròn để một ô trống nằm ở cuối, rồi cố định trạng thái ô đó ("có xô" / "không xô") ngay từ đầu DP — ô cuối vừa là "ô trước" của ô đầu tiên, vừa là ô kết thúc. Thử hai trường hợp ⇒ bài vòng tròn quy về hai bài tuyến tính.

## 20. Template nhớ nhanh + Checklist

```python
# Tham lam "phủ bằng vật đặt ở ô kề", ưu tiên phía chưa xét (bên phải)
i = ans = 0
while i < n:
    if s[i] == "H":
        if i + 1 < n and s[i + 1] == ".":
            ans += 1; i += 3; continue       # xô ở i+1 nuôi cả chuột i+2
        if i > 0 and s[i - 1] == ".":
            ans += 1
        else:
            return -1
    i += 1
return ans
# Công thức: -1 nếu có chuột bị kẹp; ngược lại s.count("H") - s.count("H.H")
```

**Checklist trước khi submit:**
- [ ] Ưu tiên đặt xô **bên phải**.
- [ ] Bỏ qua chuột đã được ăn nhờ xô trước (nhảy `i += 3` hoặc mảng `placed`).
- [ ] `-1` cho chuột bị kẹp, kể cả ở **biên** (`"H"`, `"HH."`, `".HH"`).
- [ ] Chỉ đặt xô ở ô `'.'`.
- [ ] Đếm `"H.H"` **không chồng nhau**.